输入数据准备，需要准备好json文件格式的数据，即一个庞大的向量数据库

In [1]:
import pandas as pd

# 用户历史浏览记录
user_history = pd.DataFrame({
    "title": ["Deep Learning", "Transformer in NLP", "GANs for Image Generation"],
    "author": ["Ian Goodfellow", "Vaswani et al.", "Ian Goodfellow"],
    "year": [2016, 2017, 2014]
})

# 文献数据库
literature_db = pd.DataFrame({
    "title": ["Deep Learning", "Attention is All You Need", "GANs for Image Generation",
              "Deep Reinforcement Learning", "Understanding Machine Learning"],
    "author": ["Ian Goodfellow", "Vaswani et al.", "Ian Goodfellow",
               "David Silver", "Shai Shalev-Shwartz"],
    "year": [2016, 2017, 2014, 2015, 2014]
})


阶段 1：召回
在召回阶段，我们使用简单的基于相似度的方法，例如通过文章标题和作者的相似性匹配文献

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def recall_stage(user_history, literature_db, top_k=10):
    # Combine user history into a single string for comparison
    user_history_texts = user_history["title"] + " " + user_history["author"]
    literature_texts = literature_db["title"] + " " + literature_db["author"]

    # TF-IDF 向量化
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(list(user_history_texts) + list(literature_texts))
    
    # 计算相似度
    user_vector = tfidf_matrix[:len(user_history_texts)]
    db_vector = tfidf_matrix[len(user_history_texts):]
    similarity_matrix = cosine_similarity(user_vector, db_vector)

    # 按照相似度为每篇文献打分
    scores = similarity_matrix.mean(axis=0)
    literature_db["score"] = scores

    # 返回按相似度排序的文献
    return literature_db.sort_values("score", ascending=False).head(top_k)

recalled_documents = recall_stage(user_history, literature_db)
print(recalled_documents)


                            title               author  year     score
0                   Deep Learning       Ian Goodfellow  2016  0.440561
2       GANs for Image Generation       Ian Goodfellow  2014  0.440561
3     Deep Reinforcement Learning         David Silver  2015  0.118361
1       Attention is All You Need       Vaswani et al.  2017  0.116583
4  Understanding Machine Learning  Shai Shalev-Shwartz  2014  0.043850


阶段 2：粗排
在粗排阶段，我们可以基于简单的特征，例如文章年份或历史浏览频次，构建一个逻辑回归模型。

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np

def coarse_ranking(recalled_documents, user_history):
    # 构造特征
    recalled_documents["title_len"] = recalled_documents["title"].apply(len)
    recalled_documents["year_diff"] = abs(recalled_documents["year"] - user_history["year"].mean())
    recalled_documents["author_match"] = recalled_documents["author"].apply(
        lambda x: int(x in user_history["author"].values)
    )

    # 简单的二分类标签，假设用户喜欢自己历史浏览的文章
    recalled_documents["label"] = recalled_documents["author_match"]

    # 特征和标签
    features = recalled_documents[["title_len", "year_diff", "author_match"]].values
    labels = recalled_documents["label"].values

    # 训练逻辑回归模型
    model = LogisticRegression()
    model.fit(features, labels)

    # 预测得分
    scores = model.predict_proba(features)[:, 1]
    recalled_documents["coarse_score"] = scores

    # 返回按粗排得分排序的文献
    return recalled_documents.sort_values("coarse_score", ascending=False)

coarsely_ranked_docs = coarse_ranking(recalled_documents, user_history)
print(coarsely_ranked_docs)


                            title               author  year     score  \
0                   Deep Learning       Ian Goodfellow  2016  0.440561   
2       GANs for Image Generation       Ian Goodfellow  2014  0.440561   
1       Attention is All You Need       Vaswani et al.  2017  0.116583   
3     Deep Reinforcement Learning         David Silver  2015  0.118361   
4  Understanding Machine Learning  Shai Shalev-Shwartz  2014  0.043850   

   title_len  year_diff  author_match  label  coarse_score  
0         13   0.333333             1      1      0.999990  
2         25   1.666667             1      1      0.825795  
1         25   1.333333             1      1      0.813248  
3         27   0.666667             0      0      0.316885  
4         30   1.666667             0      0      0.044083  


阶段 3：精排
在精排阶段，可以使用更复杂的模型，例如基于神经网络的排序模型，结合更多上下文特征。

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def fine_ranking(coarsely_ranked_docs, user_history):
    # 构造更复杂的特征
    coarsely_ranked_docs["title_author_len"] = coarsely_ranked_docs["title_len"] + coarsely_ranked_docs["author"].apply(len)
    coarsely_ranked_docs["year_recent"] = 2023 - coarsely_ranked_docs["year"]
    
    features = coarsely_ranked_docs[["title_len", "year_diff", "author_match", "title_author_len", "year_recent"]].values
    labels = coarsely_ranked_docs["label"].values

    # 使用随机森林进行精排
    model = RandomForestClassifier()
    model.fit(features, labels)

    # 预测精排得分
    scores = model.predict_proba(features)[:, 1]
    coarsely_ranked_docs["fine_score"] = scores

    # 返回按精排得分排序的文献
    return coarsely_ranked_docs.sort_values("fine_score", ascending=False)

finely_ranked_docs = fine_ranking(coarsely_ranked_docs, user_history)
print(finely_ranked_docs)


                            title               author  year     score  \
0                   Deep Learning       Ian Goodfellow  2016  0.440561   
1       Attention is All You Need       Vaswani et al.  2017  0.116583   
2       GANs for Image Generation       Ian Goodfellow  2014  0.440561   
3     Deep Reinforcement Learning         David Silver  2015  0.118361   
4  Understanding Machine Learning  Shai Shalev-Shwartz  2014  0.043850   

   title_len  year_diff  author_match  label  coarse_score  title_author_len  \
0         13   0.333333             1      1      0.999990                27   
1         25   1.333333             1      1      0.813248                39   
2         25   1.666667             1      1      0.825795                39   
3         27   0.666667             0      0      0.316885                39   
4         30   1.666667             0      0      0.044083                49   

   year_recent  fine_score  
0            7        0.94  
1            6  

阶段 4：重排
在重排阶段，可以基于用户历史浏览顺序，结合上下文做更个性化的推荐。

In [8]:
def reranking(finely_ranked_docs, user_history):
    # 根据历史浏览记录的作者优先排序
    finely_ranked_docs["rerank_score"] = finely_ranked_docs["fine_score"]
    finely_ranked_docs.loc[finely_ranked_docs["author"].isin(user_history["author"]), "rerank_score"] += 0.1

    return finely_ranked_docs.sort_values("rerank_score", ascending=False)

reranked_docs = reranking(finely_ranked_docs, user_history)
print(reranked_docs)


                            title               author  year     score  \
0                   Deep Learning       Ian Goodfellow  2016  0.440561   
1       Attention is All You Need       Vaswani et al.  2017  0.116583   
2       GANs for Image Generation       Ian Goodfellow  2014  0.440561   
3     Deep Reinforcement Learning         David Silver  2015  0.118361   
4  Understanding Machine Learning  Shai Shalev-Shwartz  2014  0.043850   

   title_len  year_diff  author_match  label  coarse_score  title_author_len  \
0         13   0.333333             1      1      0.999990                27   
1         25   1.333333             1      1      0.813248                39   
2         25   1.666667             1      1      0.825795                39   
3         27   0.666667             0      0      0.316885                39   
4         30   1.666667             0      0      0.044083                49   

   year_recent  fine_score  rerank_score  
0            7        0.94     

评测 Benchmark, 可以使用以下指标评测推荐系统的准确度：

准确率（Precision）：推荐文献中实际相关文献的比例。

召回率（Recall）：实际相关文献中被推荐的比例。

NDCG（Normalized Discounted Cumulative Gain）：评估排名质量。

In [9]:
from sklearn.metrics import precision_score, recall_score

def evaluate(recommended_docs, ground_truth):
    recommended_labels = recommended_docs["label"].values
    ground_truth_labels = ground_truth["label"].values

    precision = precision_score(ground_truth_labels, recommended_labels)
    recall = recall_score(ground_truth_labels, recommended_labels)

    return {"precision": precision, "recall": recall}

# 假设 ground_truth 是人工标注的真实数据
ground_truth = finely_ranked_docs.copy()
ground_truth["label"] = [1, 0, 0, 1, 0]  # 假设标注

# 评估
results = evaluate(reranked_docs, ground_truth)
print("Evaluation Results:", results)


Evaluation Results: {'precision': 0.3333333333333333, 'recall': 0.5}


In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
import torch

# 用户历史行为序列
user_history = [
    {"title": "Deep Learning", "author": "Ian Goodfellow", "year": 2016},
    {"title": "Transformer in NLP", "author": "Vaswani et al.", "year": 2017},
    {"title": "GANs for Image Generation", "author": "Ian Goodfellow", "year": 2014},
]

# 候选文献数据库
literature_db = [
    {"title": "Deep Learning", "author": "Ian Goodfellow", "year": 2016},
    {"title": "Attention is All You Need", "author": "Vaswani et al.", "year": 2017},
    {"title": "GANs for Image Generation", "author": "Ian Goodfellow", "year": 2014},
    {"title": "Deep Reinforcement Learning", "author": "David Silver", "year": 2015},
    {"title": "Understanding Machine Learning", "author": "Shai Shalev-Shwartz", "year": 2014},
]

# 转换为 DataFrame
user_history_df = pd.DataFrame(user_history)
literature_db_df = pd.DataFrame(literature_db)

# 构造用户行为序列和候选集
user_behavior = [item["title"] for item in user_history]  # 用户行为序列
candidates = [item["title"] for item in literature_db]  # 候选项

# 标签编码器
encoder = LabelEncoder()
all_titles = list(user_history_df["title"]) + list(literature_db_df["title"])
encoder.fit(all_titles)

# 编码用户行为序列和候选项
user_behavior_encoded = encoder.transform(user_behavior)
candidates_encoded = encoder.transform(candidates)

# 假设用户历史点击过的文献标签为1，其他为0
labels = [1 if title in user_behavior else 0 for title in candidates]


DIN 模型实现

In [16]:
# 定义数据集和数据加载器
class LiteratureDataset(Dataset):
    def __init__(self, user_behavior, candidates, labels, seq_len):
        self.user_behavior = user_behavior
        self.candidates = candidates
        self.labels = labels
        self.seq_len = seq_len

    def __len__(self):
        return len(self.candidates)

    def __getitem__(self, idx):
        # 将用户行为序列填充到固定长度
        padded_behavior = np.zeros(self.seq_len, dtype=np.int64)
        padded_behavior[: len(self.user_behavior)] = self.user_behavior
        return torch.tensor(padded_behavior, dtype=torch.long), \
               torch.tensor(self.candidates[idx], dtype=torch.long), \
               torch.tensor(self.labels[idx], dtype=torch.float)

# 构造 DataLoader
seq_len = 5  # 用户行为序列长度
dataset = LiteratureDataset(user_behavior_encoded, candidates_encoded, labels, seq_len)
data_loader = DataLoader(dataset, batch_size=2, shuffle=True)



In [21]:
import torch.nn as nn
import torch

class DIN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, seq_len):
        super(DIN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)  # 嵌入层
        self.attention_fc = nn.Sequential(  # 注意力网络
            nn.Linear(2 * embedding_dim, 80),
            nn.ReLU(),
            nn.Linear(80, 1)
        )
        self.output_fc = nn.Sequential(  # 输出层
            nn.Linear(2 * embedding_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        self.seq_len = seq_len

    def forward(self, user_behavior, candidate):
        # 嵌入用户行为序列和候选项
        user_behavior_emb = self.embedding(user_behavior)  # [batch_size, seq_len, embedding_dim]
        candidate_emb = self.embedding(candidate).unsqueeze(1)  # [batch_size, 1, embedding_dim]

        # 注意力机制
        repeated_candidate = candidate_emb.expand(-1, user_behavior_emb.size(1), -1)  # 动态扩展
        attention_input = torch.cat([user_behavior_emb, repeated_candidate], dim=-1)  # 拼接 [batch_size, seq_len, 2 * embedding_dim]
        attention_weights = self.attention_fc(attention_input).squeeze(-1)  # [batch_size, seq_len]
        attention_weights = torch.softmax(attention_weights, dim=1)

        # 加权求和
        user_interest = torch.sum(user_behavior_emb * attention_weights.unsqueeze(-1), dim=1)  # [batch_size, embedding_dim]

        # 拼接用户兴趣和候选项
        concat = torch.cat([user_interest, candidate_emb.squeeze(1)], dim=-1)  # [batch_size, 2 * embedding_dim]

        # 输出评分
        output = self.output_fc(concat).squeeze(-1)  # [batch_size]
        return output



In [22]:
# 模型初始化
vocab_size = len(encoder.classes_)
embedding_dim = 8
model = DIN(vocab_size, embedding_dim, seq_len)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 训练循环
epochs = 100
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for user_behavior, candidate, label in data_loader:
        optimizer.zero_grad()
        output = model(user_behavior, candidate)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {epoch_loss:.4f}")


Epoch 1/100, Loss: 1.9332
Epoch 2/100, Loss: 1.8396
Epoch 3/100, Loss: 1.7892
Epoch 4/100, Loss: 1.7293
Epoch 5/100, Loss: 1.6643
Epoch 6/100, Loss: 1.7266
Epoch 7/100, Loss: 1.5591
Epoch 8/100, Loss: 1.6192
Epoch 9/100, Loss: 1.4730
Epoch 10/100, Loss: 1.5130
Epoch 11/100, Loss: 1.4602
Epoch 12/100, Loss: 1.4048
Epoch 13/100, Loss: 1.2668
Epoch 14/100, Loss: 1.2348
Epoch 15/100, Loss: 1.1830
Epoch 16/100, Loss: 1.1145
Epoch 17/100, Loss: 1.0780
Epoch 18/100, Loss: 1.0970
Epoch 19/100, Loss: 1.0433
Epoch 20/100, Loss: 0.9141
Epoch 21/100, Loss: 0.9358
Epoch 22/100, Loss: 0.8214
Epoch 23/100, Loss: 0.7740
Epoch 24/100, Loss: 0.7826
Epoch 25/100, Loss: 0.6739
Epoch 26/100, Loss: 0.6027
Epoch 27/100, Loss: 0.5812
Epoch 28/100, Loss: 0.5475
Epoch 29/100, Loss: 0.5623
Epoch 30/100, Loss: 0.5219
Epoch 31/100, Loss: 0.4540
Epoch 32/100, Loss: 0.4438
Epoch 33/100, Loss: 0.3840
Epoch 34/100, Loss: 0.3246
Epoch 35/100, Loss: 0.2874
Epoch 36/100, Loss: 0.2715
Epoch 37/100, Loss: 0.2604
Epoch 38/1

In [23]:
# 使用模型对候选文献进行评分
model.eval()
with torch.no_grad():
    user_behavior_input = torch.tensor(user_behavior_encoded, dtype=torch.long).unsqueeze(0).expand(len(candidates_encoded), -1)
    candidate_input = torch.tensor(candidates_encoded, dtype=torch.long)
    scores = model(user_behavior_input, candidate_input).numpy()

# 将评分与候选文献绑定
literature_db_df["din_score"] = scores
recommended_docs = literature_db_df.sort_values("din_score", ascending=False)

print(recommended_docs)


                            title               author  year  din_score
0                   Deep Learning       Ian Goodfellow  2016   0.837148
2       GANs for Image Generation       Ian Goodfellow  2014   0.534517
1       Attention is All You Need       Vaswani et al.  2017   0.002180
3     Deep Reinforcement Learning         David Silver  2015   0.001819
4  Understanding Machine Learning  Shai Shalev-Shwartz  2014   0.001576


In [24]:
from sklearn.metrics import precision_score, recall_score

def evaluate_model(predicted_scores, true_labels, top_k=5):
    top_k_indices = np.argsort(predicted_scores)[-top_k:]
    top_k_pred = [1 if i in top_k_indices else 0 for i in range(len(predicted_scores))]
    precision = precision_score(true_labels, top_k_pred)
    recall = recall_score(true_labels, top_k_pred)
    return {"precision": precision, "recall": recall}

# 评估 DIN 模型
true_labels = np.array(labels)
din_evaluation = evaluate_model(scores.flatten(), true_labels)
print("DIN Evaluation:", din_evaluation)


DIN Evaluation: {'precision': 0.4, 'recall': 1.0}
